In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx
import finufft

import cmasher as cmr
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

pa.settings.strict_units = False

from paicos.util import get_index_of_radial_range as RadialRange

In [2]:
# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 
                   305, basename='snapshot')
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 200)
# snap = pa.Snapshot('/scratch/lperrone/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 
                   # 305, basename='snapshot')
center = snap.Cat.Group['GroupPos'][0]

# radius = 1e2*snap.length
radius = 1e-2*snap.Cat.Group['Group_R_Crit200'][0]

# widths = np.array([2e2, 2e2, 2e2], dtype=float)
widths = np.ones(3)*2*radius.value

Gauss = snap.uq('G')
arepo_length = snap.length
density_unit = snap['0_Density'].uq
L_box = (arepo_length*(widths[0]))

In [3]:
all_pos = np.zeros((0,3))
all_mass = np.zeros(0)
all_hsml = np.zeros(0)
all_types = np.zeros(0, dtype='int')
for parttype in range(0,5):
    idx_sphere = RadialRange(snap[f"{parttype}_Coordinates"],
                       center, 0*radius, radius)
    all_pos = np.concatenate((all_pos, snap[f"{parttype}_Coordinates"][idx_sphere]), axis=0)
    all_mass = np.concatenate((all_mass, snap[f"{parttype}_Masses"][idx_sphere]), axis=0)
    if (parttype==0):
        hsml = np.cbrt((snap[f"{parttype}_Volume"]) / (4.0 * np.pi / 3.0)).to('arepo_length')
    else:
        ## this is wrong!!!
        # hsml = snap[f'{parttype}_SubfindHsml']
        ## this is correct
        soft_type = snap.Parameters[f'SofteningTypeOfPartType{parttype}']
        softening_comoving = (snap.Parameters[f'SofteningComovingType{soft_type}']*snap.h)*snap.length
        softening_max = (snap.Parameters[f'SofteningMaxPhysType{soft_type}']/snap.a*snap.h)*snap.length
        hsml = np.ones(snap[f"{parttype}_Coordinates"].shape[0])*min([softening_comoving, softening_max])
    all_hsml = np.concatenate((all_hsml, hsml[idx_sphere]), axis=0)
    all_types = np.concatenate((all_types, np.ones(np.sum(idx_sphere), dtype='int')*parttype))

Attempting to get derived variable: 0_Volume...	[DONE]

Attempting to get derived variable: 1_Masses...	[DONE]

Attempting to get derived variable: 3_Masses...	[DONE]



In [12]:

pe = tc.PotentialEnergy(snap, center, widths, 
                        pos=all_pos, mass=all_mass, smoothing_length=all_hsml, npix=64)

In [13]:
np.sum(pe.tile.particles_per_tile)

array(498044)

In [14]:
pe.compute_potential(angle=15)

<PaicosQuantity -7.21891422e-07 m3 arepo_mass2 / (arepo_length kg small_a small_h s2)>

In [ ]:
U = []
result = %timeit -n1 -r1 -o U.append(pe.compute_potential(angle=15))

In [ ]:
U

In [9]:
from numba import njit, prange
import cupy as cp
from numba import cuda
import math


@cuda.jit()
def cuda_potential(masses, pos, hsml, potential):

    Np =  masses.shape[0]
    i = cuda.grid(1)

    if (i < Np):
        for j in range(Np):
            h = max(hsml[i], hsml[j])
            m1m2 = masses[i]*masses[j]
            if (i==j):
                potential[i] -= m1m2*(14./5.)/h
            else:
                r = math.sqrt((pos[i][0] - pos[j][0])**2 +
                             (pos[i][1] - pos[j][1])**2 +
                             (pos[i][2] - pos[j][2])**2)
                potential_tmp = m1m2/r
                
                # if (r <= h):
                #     potential_tmp *= (0.5*r/h)*(3-(r/h)**2)
                u = r/h
                if (u < 0.5):
                    potential_tmp *= (14./5.)*u - (16./3.)*u**3 + (48./5.)*u**5 - (32./5.)*u**6
                elif (u >= 0.5 and u < 1):
                    potential_tmp *= 16.*u**4 - (48./5.)*u**5 + (32./15.)*u**6
                
                potential[i] -= potential_tmp
                


def direct_calculation_cuda(snap, positions, masses, smoothing_length):
    # snap_select = snap.select(index, parttype=0)

    Np = masses.shape[0]

    if isinstance(smoothing_length.value, np.ndarray):
        assert smoothing_length.shape == masses.shape
    else:
        smoothing_length = np.ones(Np) * smoothing_length

    mass = cp.array(masses.value)
    pos = cp.array(positions.value)
    hsml = cp.array(smoothing_length.value)
    potential = cp.zeros_like(mass)

    threadsperblock = 256
    blocks_1d = (Np + (threadsperblock - 1)) // threadsperblock
    

    cuda_potential[blocks_1d, threadsperblock](mass, pos, hsml, potential)

    potential = cp.asnumpy(cp.sum(potential))
    
    G = pa.astropy.constants.G
    potential = 0.5 * G * potential * snap.mass**2 / snap.length
    return potential
    

In [10]:
direct_calculation_cuda(snap, all_pos, 
                            all_mass, all_hsml)

<PaicosQuantity -7.22504834e-07 m3 arepo_mass2 / (arepo_length kg small_a small_h s2)>

In [ ]:
# U_bench = []
# runtime_bench = [] # in s
# for angle in angles:
#     U_tmp = []
#     result = %timeit -n1 -r5 -o U_tmp.append(pe.compute_potential(smoothing_length, angle=angle))
#     U_bench.append(U_tmp[0])
#     runtime_bench.append(result.average)

In [ ]:
# runtime_bench

In [ ]:
# U_direct = []
# # runtime_direct = [] # in s
# result = %timeit -n1 -r5 -o U_direct.append(direct_calculation_cuda(snap, snap['0_Coordinates'][pe.index], snap['0_Masses'][pe.index], smoothing_length[pe.index]))

# U_direct = U_direct[0]
# runtime_direct = result.average

In [ ]:
# U_direct

In [ ]:
# U_bench

In [ ]:
# relative_err = np.array([np.abs((U_bench[i]-U_direct)/U_direct).value for i in range(len(U_bench))])

In [ ]:


# fig, ax = plt.subplots(2, 1, sharex=True)

# ax[0].plot(angles, relative_err*100, ls='', marker='o', label='tiles')
# ax[1].plot(angles, runtime_bench, ls='', marker='o', label='tiles')
# ax[1].hlines(runtime_direct, xmin=-5, xmax=45, color='orange', ls='-', label='direct N^2')

# ax[0].text(20, 0.02, f'direct N^2: {U_direct.value:.6f}')
# ax[0].text(20, 0.015, f'theory: {U_theory.value:.6f}')

# ax[0].set_xlim(xmin=-5, xmax=45)

# ax[1].set_xlabel('angle')
# ax[0].set_ylabel('relative err [%]')
# ax[1].set_ylabel('timings [s]')
# ax[0].legend()
# ax[1].legend()

# fig.suptitle('Benchmark of gravitational potential for uniform sphere')
# plt.savefig('./uniform_sphere.pdf')

# plt.show()